In [ ]:
"""
SiPM IV Curve Analysis Tool

Analyzes current-voltage (IV) characteristics of Silicon Photomultipliers (SiPMs)
and determines breakdown voltages using Landau distribution fitting.

Usage:
    python sipm_IV_curves.py <path_to_folder>

The folder should contain .txt files with voltage and current data.
"""

import os
import re
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.optimize import curve_fit
from scipy.stats import moyal


def first_number_file(name: str) -> int:
    """
    Extract the first number from a filename.

    Args:
        name (str): Filename to parse

    Returns:
        int or None: First number found in the filename, or None if no number exists
    """
    match = re.search(r'\d+', name)
    if match:
        result = int(match.group())
    else:
        result = None
        print(f'Warning: No number found in {name}.')
    return result


def find_key(file: str) -> str:
    """
    Find the first key in an HDF5 file.

    Args:
        file (str): Path to HDF5 file

    Returns:
        str: First key in the HDF5 file
    """
    with h5py.File(file, 'r') as f:
        key = next(iter(f.keys()))
        print("hdf5 keys:", list(f.keys()))
    return key


def natural_key(text: str) -> list:
    """
    Generate a key for natural sorting (handles numbers in strings correctly).

    Args:
        text (str): String to generate sort key for

    Returns:
        list: Sort key that handles numeric and text parts separately

    Example:
        ['file1.txt', 'file10.txt', 'file2.txt'] -> ['file1.txt', 'file2.txt', 'file10.txt']
    """
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', text)]


def plot_IV_overlay(file_names: list, log: bool = False, save_dir: str = None):
    """
    Plot overlaid IV curves from multiple data files.

    Args:
        file_names (list): List of file paths containing IV data
        log (bool): If True, use logarithmic scale for current axis
        save_dir (str): Directory to save the plot (if None, no saving)
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    for file in file_names:
        sipm_n = first_number_file(file)

        # Load and parse data
        df = pd.read_csv(file, sep="\t")
        df.columns = ["Voltage", "Current"]

        df["Voltage"] = pd.to_numeric(df["Voltage"], errors='coerce')
        df["Current"] = pd.to_numeric(df["Current"], errors='coerce')

        # Plot with markers at regular intervals
        ax.plot(df["Voltage"], df["Current"],
                marker='o',
                markersize=4,
                markevery=8,
                linewidth=0,
                label=sipm_n)

    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("Current (mA)")
    ax.set_title("Voltage vs Current")
    if log:
        ax.set_yscale('log')
    ax.grid(True)
    ax.legend()

    plt.tight_layout()

    # Save as PNG if save_dir is provided
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "IV_overlay.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show(block=True)
    plt.close()


def landau(x: np.array,
           A: float,
           mu: float,
           sigma: float):
    """
    Landau distribution using scipy's Moyal approximation.

    Args:
        x (array): Input values
        A (float): Amplitude parameter
        mu (float): Location parameter (peak position)
        sigma (float): Scale parameter (width)

    Returns:
        array: Landau distribution values
    """
    return A * moyal.pdf(x, loc=mu, scale=sigma)


def analyze_breakdown(file_names: list,
                      window: float = 0.2,
                      save_dir: str = None) -> list:
    """
    Analyze breakdown voltage from IV curves using derivative method.

    The breakdown voltage is identified by fitting a Landau distribution to
    the quantity (1/I) * (dI/dV), which peaks at the breakdown point.

    Args:
        file_names (list): List of file paths containing IV data
        window (float): Half-width (in volts) around the mean for xlim (currently unused)
        save_dir (str): Directory to save individual plots (if None, no saving)

    Returns:
        list: List of tuples (sipm_id, breakdown_voltage, voltage_error)
    """
    results = []

    for file in file_names:
        # Load data
        sipm_n = first_number_file(file)
        df = pd.read_csv(file, sep="\t")
        df.columns = ["Voltage", "Current"]
        V = df["Voltage"].values
        I = df["Current"].values

        # Calculate derivative dI/dV
        dIdV = np.gradient(I, V)

        # Calculate breakdown indicator: (1/I) * (dI/dV)
        # This quantity peaks sharply at breakdown voltage
        y = (1 / I) * dIdV

        # Remove last two points to avoid edge effects
        V = V[:-2]
        y = y[:-2]

        # Initial parameter guesses for Landau fit
        A_guess = 0
        mu_guess = V[np.argmax(y)]
        sigma_guess = (np.max(V) - np.min(V)) / 10

        fit_success = True
        try:
            # Attempt Landau fit
            popt, pcov = curve_fit(
                landau, V, y,
                p0=[A_guess, mu_guess, sigma_guess],
                maxfev=10000)
            A, mu, sigma = popt
            mu_err = np.sqrt(pcov[1, 1])
        except Exception:
            # Fallback if fit fails: use peak position with estimated error
            fit_success = False
            mu = V[np.argmax(y)]
            mu_err = 0.03

        results.append((str(sipm_n), mu, mu_err))

        # --- Plot breakdown analysis for this file ---
        plt.figure()
        plt.plot(V, y, 'o', label="(1/I) * (dI/dV)")

        # Overlay Landau fit if successful
        if fit_success:
            V_fit = np.linspace(min(V), max(V), 500)
            plt.plot(V_fit, landau(V_fit, *popt), label="Landau fit")

        # Set plot limits
        plt.xlim(49, 53)
        plt.yscale('log')
        plt.ylim(1e-6, 1e2)

        plt.xlabel("Voltage (V)")
        plt.ylabel("(1/I) * (dI/dV)")
        plt.title(f"Breakdown analysis: {file}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        # Save as PNG if save_dir is provided
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"sipm_{sipm_n}_breakdown.png")
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")

        plt.show(block=True)
        plt.close()

    return results


def plot_breakdown_summary(results: list,
                           ylim: tuple = None,
                           save_dir: str = None):
    """
    Plot summary of breakdown voltages for all analyzed SiPMs.

    Results are sorted by breakdown voltage and displayed with error bars.

    Args:
        results (list): List of tuples (sipm_id, breakdown_voltage, voltage_error)
        ylim (tuple): Optional y-axis limits as (ymin, ymax)
        save_dir (str): Directory to save the plot (if None, no saving)
    """
    # Sort results by breakdown voltage (ascending)
    results_sorted = sorted(results, key=lambda r: r[1])

    # Extract sorted values
    names = [r[0] for r in results_sorted]
    mus = [r[1] for r in results_sorted]
    mu_errs = [r[2] for r in results_sorted]

    x = np.arange(len(names))

    # Create plot with error bars
    plt.figure()
    plt.errorbar(x, mus, yerr=mu_errs, fmt='o', capsize=5)

    # Configure x-axis
    plt.xticks(x, names, rotation=45, ha='right')

    if ylim:
        plt.ylim(ylim)

    plt.xlabel("Dataset")
    plt.ylabel("Breakdown Voltage (V)")
    plt.title("Breakdown Voltage (Sorted)")
    plt.grid(True)

    plt.tight_layout()

    # Save as PNG if save_dir is provided
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "breakdown_summary.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show(block=True)
    plt.close()


def main():
    """
    Main entry point for the SiPM IV curve analysis tool.
    """
    # Hardcoded path - CHANGE THIS IF NEEDED
    folder_path = r"C:\Users\leaga\Desktop\Internship UoM\SiPMs IV Curves"

    # Validate folder path
    if not os.path.isdir(folder_path):
        print(f"Error: '{folder_path}' is not a valid directory")
        return 1

    # Create output directory for saved plots
    plots_dir = os.path.join(folder_path, "analysis_plots")
    os.makedirs(plots_dir, exist_ok=True)
    print(f"Saving plots to: {plots_dir}")

    # Find all .txt files and sort naturally
    file_names = [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.endswith('.txt')]

    file_names = sorted(file_names, key=natural_key)

    if not file_names:
        print(f"Error: No .txt files found in '{folder_path}'")
        return 1

    print(f"Found {len(file_names)} data files")
    print("Generating IV overlay plot...")

    # Generate plots with saving enabled
    plot_IV_overlay(file_names, save_dir=plots_dir)

    print("Analyzing breakdown voltages...")
    results = analyze_breakdown(file_names, save_dir=plots_dir)

    print("\nBreakdown voltage results:")
    for sipm_id, vbd, vbd_err in results:
        print(f"  SiPM {sipm_id}: {vbd:.3f} ± {vbd_err:.3f} V")

    print("\nGenerating summary plot...")
    plot_breakdown_summary(results, save_dir=plots_dir)

    print(f"\nAnalysis complete! All plots saved to: {plots_dir}")
    return 0


if __name__ == "__main__":
    exit(main())